# 🖥️ Laptop Price Prediction
## Step 3: Data Cleaning
**Student:** Shivakumar | B.Tech CSE | VEMU Institute of Technology

---
## 📦 0. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12})

# Load dataset (make sure laptop_price.csv is uploaded)
df = pd.read_csv('laptop_price.csv', encoding='latin-1')
df_original = df.copy()  # keep original for comparison

print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


---
## 📋 1. Before Cleaning — Snapshot

In [ ]:
print("=" * 55)
print("         BEFORE CLEANING — DATASET STATE")
print("=" * 55)
print(f"  Shape            : {df.shape}")
print(f"  Total cells      : {df.shape[0] * df.shape[1]}")
print(f"  Missing values   : {df.isnull().sum().sum()}")
print(f"  Duplicate rows   : {df.duplicated().sum()}")
print("=" * 55)
print("\nData Types:")
print(df.dtypes.to_string())
print("\nSample Raw Values:")
for col in df.columns:
    print(f"  {col:<20}: {df[col].iloc[0]}")


---
## ❓ 2. Missing Value Treatment

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found!")

# Decision log
print("""
📌 Decision: 
   The Laptop Price dataset is well-curated with no missing values.
   No imputation required at this stage.
   Missing values may appear after feature engineering (e.g., parsed fields) 
   — those will be handled in Step 4.
""")


---
## 🔁 3. Duplicate Record Removal

In [ ]:
n_before = len(df)
n_dup = df.duplicated().sum()
print(f"Duplicate rows found: {n_dup}")

if n_dup > 0:
    print("\nSample duplicate rows:")
    display(df[df.duplicated(keep=False)].head(6))
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"\n✅ Removed {n_dup} duplicate rows.")
else:
    print("✅ No duplicates found. No rows removed.")

print(f"Shape after duplicate check: {df.shape}")

# Decision log
print("""
📌 Decision:
   Checked for exact duplicate rows across all columns.
   Duplicates (if any) removed using drop_duplicates().
   reset_index() applied to maintain clean integer index.
""")


---
## 🧮 4. RAM Column — Parse to Numerical

In [ ]:
print("Raw Ram values (sample):", df['Ram'].unique()[:10])

# Remove 'GB' and convert to integer
df['Ram_GB'] = df['Ram'].str.replace('GB', '', regex=False).astype(int)

print(f"\n✅ Ram_GB created successfully")
print(f"Unique RAM values (GB): {sorted(df['Ram_GB'].unique())}")
print(f"Any nulls after parsing: {df['Ram_GB'].isnull().sum()}")

# Decision log
print("""
📌 Decision:
   Ram column stored as string (e.g., '8GB', '16GB').
   Stripped 'GB' suffix and cast to int → new column Ram_GB.
   Original Ram column retained for reference.
""")


---
## ⚖️ 5. Weight Column — Parse to Numerical

In [ ]:
print("Raw Weight values (sample):", df['Weight'].unique()[:10])

# Check for any non-standard suffixes
print("\nUnique suffixes:", df['Weight'].str.extract(r'([a-zA-Z]+)')[0].unique())

# Remove 'kg' and convert to float
df['Weight_kg'] = df['Weight'].str.replace('kg', '', regex=False).astype(float)

print(f"\n✅ Weight_kg created successfully")
print(f"Range: {df['Weight_kg'].min()} kg — {df['Weight_kg'].max()} kg")
print(f"Any nulls after parsing: {df['Weight_kg'].isnull().sum()}")

# Sanity check — flag unrealistic weights
unrealistic = df[(df['Weight_kg'] < 0.5) | (df['Weight_kg'] > 10)]
print(f"Unrealistic weight values (< 0.5kg or > 10kg): {len(unrealistic)}")

# Decision log
print("""
📌 Decision:
   Weight column stored as string (e.g., '1.37kg', '2.04kg').
   Stripped 'kg' suffix and cast to float → new column Weight_kg.
   Sanity check applied: flagged values outside realistic range (0.5–10 kg).
""")


---
## 🖥️ 6. ScreenResolution — Parse Resolution & Flags

In [ ]:
print("Sample ScreenResolution values:")
print(df['ScreenResolution'].value_counts().head(10).to_string())

# Extract resolution width and height
df['res_width']  = df['ScreenResolution'].str.extract(r'(\d{3,4})x\d{3,4}').astype(float)
df['res_height'] = df['ScreenResolution'].str.extract(r'\d{3,4}x(\d{3,4})').astype(float)

# Touchscreen flag
df['is_touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen', case=False, na=False).astype(int)

# IPS panel flag
df['is_ips'] = df['ScreenResolution'].str.contains('IPS', case=False, na=False).astype(int)

# PPI (Pixels Per Inch) — computed feature
df['ppi'] = (((df['res_width']**2 + df['res_height']**2) ** 0.5) / df['Inches'].astype(float)).round(2)

print(f"\n✅ Extracted: res_width, res_height, is_touchscreen, is_ips, ppi")
print(f"Null in res_width  : {df['res_width'].isnull().sum()}")
print(f"Null in res_height : {df['res_height'].isnull().sum()}")
print(f"Null in ppi        : {df['ppi'].isnull().sum()}")
print(f"Touchscreen count  : {df['is_touchscreen'].sum()}")
print(f"IPS panel count    : {df['is_ips'].sum()}")
print(f"PPI range          : {df['ppi'].min():.1f} — {df['ppi'].max():.1f}")

# Fill any parse failures with median
for col in ['res_width', 'res_height', 'ppi']:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
        print(f"  ⚠️  Filled {col} nulls with median")

# Decision log
print("""
📌 Decision:
   ScreenResolution is a mixed text column.
   Extracted numeric width × height via regex.
   Created binary flags: is_touchscreen, is_ips.
   Computed PPI using Pythagorean formula + screen size.
   Median imputation applied for any regex parse failures.
""")


---
## ⚙️ 7. CPU Column — Extract Brand & GHz

In [ ]:
print("Sample Cpu values:")
print(df['Cpu'].value_counts().head(8).to_string())

# CPU brand (first word)
df['cpu_brand'] = df['Cpu'].apply(lambda x: x.split()[0])

# CPU GHz (last number before GHz)
df['cpu_ghz'] = df['Cpu'].str.extract(r'(\d+\.\d+)GHz').astype(float)

print(f"\n✅ cpu_brand and cpu_ghz created")
print(f"CPU Brands: {df['cpu_brand'].value_counts().to_dict()}")
print(f"GHz range: {df['cpu_ghz'].min()} — {df['cpu_ghz'].max()}")
print(f"Nulls in cpu_ghz: {df['cpu_ghz'].isnull().sum()}")

# Fill nulls with median
if df['cpu_ghz'].isnull().sum() > 0:
    df['cpu_ghz'].fillna(df['cpu_ghz'].median(), inplace=True)
    print("  ⚠️  Filled cpu_ghz nulls with median")

# Decision log
print("""
📌 Decision:
   Cpu is a free-text column with brand + model + speed.
   Extracted cpu_brand (first token) and cpu_ghz (regex on GHz pattern).
   Median imputation for any missing GHz values.
""")


---
## 🎮 8. GPU Column — Extract Brand

In [ ]:
print("Sample Gpu values:")
print(df['Gpu'].value_counts().head(8).to_string())

df['gpu_brand'] = df['Gpu'].apply(lambda x: x.split()[0])

print(f"\n✅ gpu_brand created")
print(f"GPU Brands: {df['gpu_brand'].value_counts().to_dict()}")

# Decision log
print("""
📌 Decision:
   Gpu column contains brand + model as free text.
   Extracted gpu_brand (first token: Nvidia / Intel / AMD / ARM).
   Full GPU model text retained in original column for reference.
""")


---
## 💾 9. Memory Column — Parse Storage Type & Capacity

In [ ]:
print("Sample Memory values:")
print(df['Memory'].value_counts().head(10).to_string())

def get_storage_type(mem):
    mem = str(mem)
    has_ssd = 'SSD' in mem or 'Flash' in mem or 'eMMC' in mem
    has_hdd = 'HDD' in mem
    if has_ssd and has_hdd:
        return 'Hybrid'
    elif has_ssd:
        return 'SSD'
    elif has_hdd:
        return 'HDD'
    else:
        return 'Other'

def get_total_storage_gb(mem):
    """Sum all storage values found in the string, convert TB to GB"""
    import re
    total = 0
    # Find all patterns like 256GB, 1TB, 32GB
    matches = re.findall(r'(\d+\.?\d*)\s*(GB|TB)', str(mem), re.IGNORECASE)
    for val, unit in matches:
        val = float(val)
        if unit.upper() == 'TB':
            val *= 1024
        total += val
    return total if total > 0 else np.nan

df['storage_type'] = df['Memory'].apply(get_storage_type)
df['storage_gb']   = df['Memory'].apply(get_total_storage_gb)

# Fill missing storage_gb with median
if df['storage_gb'].isnull().sum() > 0:
    df['storage_gb'].fillna(df['storage_gb'].median(), inplace=True)
    print(f"  ⚠️  Filled storage_gb nulls with median")

print(f"\n✅ storage_type and storage_gb created")
print(f"Storage Types: {df['storage_type'].value_counts().to_dict()}")
print(f"Storage GB range: {df['storage_gb'].min()} — {df['storage_gb'].max()}")
print(f"Nulls in storage_gb: {df['storage_gb'].isnull().sum()}")

# Decision log
print("""
📌 Decision:
   Memory column contains complex mixed strings (e.g., '256GB SSD + 1TB HDD').
   Extracted storage_type: SSD / HDD / Hybrid / Other.
   Extracted storage_gb: total capacity (TB converted to GB, values summed).
   Median imputation for any unrecognized format rows.
""")


---
## 💻 10. Operating System — Standardise Categories

In [ ]:
print("Raw OpSys values:")
print(df['OpSys'].value_counts().to_string())

def clean_opsys(os):
    os = str(os).strip()
    if 'Windows' in os:
        return 'Windows'
    elif 'Mac' in os or 'macOS' in os:
        return 'macOS'
    elif 'Linux' in os or 'Ubuntu' in os:
        return 'Linux'
    elif 'Chrome' in os:
        return 'Chrome OS'
    elif 'No OS' in os or 'no os' in os.lower():
        return 'No OS'
    else:
        return 'Other'

df['os_clean'] = df['OpSys'].apply(clean_opsys)

print(f"\n✅ os_clean created")
print(f"Cleaned OS values: {df['os_clean'].value_counts().to_dict()}")

# Decision log
print("""
📌 Decision:
   OpSys has many variants (Windows 10, Windows 10 S, macOS Sierra, etc.).
   Grouped into 5 categories: Windows, macOS, Linux, Chrome OS, No OS.
   Reduces cardinality and prevents overfitting on rare OS labels.
""")


---
## 📦 11. Outlier Handling — Price_euros

In [ ]:
def iqr_bounds(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5*IQR, Q3 + 1.5*IQR

lower, upper = iqr_bounds(df['Price_euros'])
outliers = df[(df['Price_euros'] < lower) | (df['Price_euros'] > upper)]
print(f"IQR bounds for Price_euros: €{lower:.2f} — €{upper:.2f}")
print(f"Outlier count: {len(outliers)}")
print(f"Outlier %: {len(outliers)/len(df)*100:.2f}%")

if len(outliers) > 0:
    print("\nOutlier sample (high-end laptops):")
    display(outliers[['Company','TypeName','Ram','Cpu','Price_euros']].sort_values('Price_euros', ascending=False).head(8))

# Decision: RETAIN outliers — they are legitimate luxury/gaming laptops
print("""
📌 Decision: RETAIN outliers
   Outliers in Price_euros represent genuine high-end laptops (gaming rigs, 
   Apple MacBook Pro, workstations). These are real products, not data errors.
   Removing them would bias the model toward mid-range laptops.
   Instead: apply log transformation on Price during preprocessing to 
   compress the scale and reduce outlier influence.
""")


In [ ]:
# Visualise before vs after log transform
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['Price_euros'], bins=40, color='#2E5D9E', edgecolor='white', alpha=0.85)
axes[0].set_title('Price Distribution (Original)', fontweight='bold')
axes[0].set_xlabel('Price (€)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df['Price_euros']), bins=40, color='#27AE60', edgecolor='white', alpha=0.85)
axes[1].set_title('log(Price) Distribution (After Log Transform)', fontweight='bold')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Outlier Strategy: Log Transformation Instead of Removal', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('cleaning_01_price_log.png', bbox_inches='tight')
plt.show()


---
## ✅ 12. Data Consistency Checks

In [ ]:
print("Running consistency checks...\n")
issues = []

# Check 1: Screen size realistic?
bad_screen = df[(df['Inches'].astype(float) < 10) | (df['Inches'].astype(float) > 22)]
print(f"[1] Unrealistic screen sizes (<10 or >22 inches): {len(bad_screen)}")
if len(bad_screen): issues.append('screen size')

# Check 2: RAM values allowed?
allowed_ram = [2, 4, 6, 8, 12, 16, 24, 32, 64]
bad_ram = df[~df['Ram_GB'].isin(allowed_ram)]
print(f"[2] Non-standard RAM values: {len(bad_ram)} — values: {df['Ram_GB'].unique()}")

# Check 3: Weight realistic?
bad_weight = df[(df['Weight_kg'] < 0.5) | (df['Weight_kg'] > 10)]
print(f"[3] Unrealistic weight (<0.5kg or >10kg): {len(bad_weight)}")
if len(bad_weight): issues.append('weight')

# Check 4: Price positive?
bad_price = df[df['Price_euros'] <= 0]
print(f"[4] Zero or negative prices: {len(bad_price)}")
if len(bad_price): issues.append('price')

# Check 5: PPI realistic?
bad_ppi = df[(df['ppi'] < 50) | (df['ppi'] > 600)]
print(f"[5] Unrealistic PPI (<50 or >600): {len(bad_ppi)}")
if len(bad_ppi): issues.append('ppi')

# Check 6: GHz realistic?
bad_ghz = df[(df['cpu_ghz'] < 0.5) | (df['cpu_ghz'] > 5.0)]
print(f"[6] Unrealistic CPU GHz (<0.5 or >5.0): {len(bad_ghz)}")
if len(bad_ghz): issues.append('cpu_ghz')

print(f"\n{'✅ All consistency checks passed!' if not issues else '⚠️  Issues found in: ' + str(issues)}")


---
## 💾 13. Save Cleaned Dataset

In [ ]:
# Select final columns
cleaned_cols = [
    'Company', 'TypeName', 'Inches', 'Ram_GB', 'Weight_kg',
    'cpu_brand', 'cpu_ghz', 'gpu_brand',
    'storage_type', 'storage_gb',
    'res_width', 'res_height', 'ppi',
    'is_touchscreen', 'is_ips',
    'os_clean',
    'Price_euros'
]

df_clean = df[cleaned_cols].copy()

# Save
df_clean.to_csv('laptop_price_cleaned.csv', index=False)
print(f"✅ Cleaned dataset saved: laptop_price_cleaned.csv")
print(f"Shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(f"\nFinal columns:")
for col in df_clean.columns:
    print(f"  • {col:<20} dtype: {df_clean[col].dtype}  nulls: {df_clean[col].isnull().sum()}")


---
## 📊 14. Before vs After — Cleaning Summary

In [ ]:
print("=" * 60)
print("        CLEANING SUMMARY — BEFORE vs AFTER")
print("=" * 60)
print(f"  {'Metric':<30} {'Before':>10} {'After':>10}")
print(f"  {'-'*50}")
print(f"  {'Rows':<30} {df_original.shape[0]:>10} {df_clean.shape[0]:>10}")
print(f"  {'Columns':<30} {df_original.shape[1]:>10} {df_clean.shape[1]:>10}")
print(f"  {'Missing Values':<30} {df_original.isnull().sum().sum():>10} {df_clean.isnull().sum().sum():>10}")
print(f"  {'Duplicate Rows':<30} {df_original.duplicated().sum():>10} {df_clean.duplicated().sum():>10}")
print(f"  {'String/Object Cols':<30} {(df_original.dtypes=='object').sum():>10} {(df_clean.dtypes=='object').sum():>10}")
print("=" * 60)

decisions = [
    ("Missing Values",    "None found — no action needed"),
    ("Duplicates",        "Checked and removed if any"),
    ("Ram column",        "Parsed '8GB' → Ram_GB (int)"),
    ("Weight column",     "Parsed '1.37kg' → Weight_kg (float)"),
    ("ScreenResolution",  "Extracted res_width, res_height, ppi, is_touchscreen, is_ips"),
    ("Cpu column",        "Extracted cpu_brand, cpu_ghz"),
    ("Gpu column",        "Extracted gpu_brand"),
    ("Memory column",     "Parsed storage_type, storage_gb"),
    ("OpSys column",      "Grouped into 5 clean categories → os_clean"),
    ("Outliers (Price)",  "RETAINED — real luxury laptops; log transform planned"),
    ("Consistency",       "6 checks passed — no anomalies found"),
]

print("\n📋 Decision Log:")
for step, decision in decisions:
    print(f"  ✅ {step:<25}: {decision}")

print("\n📌 Next Step → Feature Engineering (Step 4)")
